# G-reasoner smoke test (clean rebuild)

**How to use:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.

Every cell runs top to bottom. Nothing to skip. Colab wipes `/content` after a few days, so on a fresh runtime just Run all again.

Success = the last cell prints `=== G-reasoner loaded successfully ===` and two Apple documents with scores.

## 1. Install Python 3.12
gfmrag needs Python 3.12, so it lives in its own venv at `/content/py312`.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq python3.12 python3.12-venv
!python3.12 --version

## 2. Create the venv and install gfmrag + ninja
Slow (several minutes).

In [ ]:
!python3.12 -m venv /content/py312
!/content/py312/bin/python -m pip install -q --upgrade pip
!/content/py312/bin/python -m pip install -q gfmrag ninja
!/content/py312/bin/python -c "import gfmrag, ninja; print('gfmrag + ninja imported successfully')"

## 3. Put ninja on PATH and define a helper
The notebook kernel can't import gfmrag, so every gfmrag step runs through the venv's Python with `run_py`. It passes PATH explicitly so ninja is always found.

In [ ]:
import os, subprocess

PYTHON = "/content/py312/bin/python"
os.environ["PATH"] = "/content/py312/bin:" + os.environ["PATH"]

def run_py(script):
    result = subprocess.run(
        [PYTHON, "-c", script],
        capture_output=True,
        text=True,
        env=os.environ.copy(),
    )
    print("--- STDOUT ---")
    print(result.stdout)
    if result.returncode != 0:
        print("--- STDERR (last part) ---")
        print(result.stderr[-4000:])
        print(f"FAILED with return code {result.returncode}")
    return result

print("ninja:", subprocess.run(["which", "ninja"], capture_output=True, text=True).stdout)

## 4. Patch gfmrag's PyG version parsing
gfmrag crashes on PyG versions like `2.8.0.post1`. This keeps only the first three parts.

In [ ]:
patch = r"""
from pathlib import Path
path = Path("/content/py312/lib/python3.12/site-packages/gfmrag/models/ultra/layers.py")
text = path.read_text()
old = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]'
new = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")[:3]]'
if old in text:
    path.write_text(text.replace(old, new))
    print("Patch applied successfully.")
elif new in text:
    print("Patch already applied.")
else:
    print("Original line not found. Check layers.py manually.")
for line in path.read_text().splitlines():
    if "pyg_version =" in line:
        print("Patched line:", line.strip())
"""
run_py(patch);

## 5. Create the toy finance graph
Three CSVs in `data/toy_fin/processed/stage1/`. Because these exist, G-reasoner loads them directly and never builds its own graph.

In [ ]:
import os
import pandas as pd

d = "data/toy_fin/processed/stage1"
os.makedirs(d, exist_ok=True)

pd.DataFrame([
    ["Apple Inc.", "entity", "{}"],
    ["Apple Operations International", "entity", "{}"],
    ["Ireland", "entity", "{}"],
    ["AAPL_2025_ex21_c1", "document", "{}"],
    ["AAPL_2025_item1_c3", "document", "{}"],
], columns=["name", "type", "attributes"]).to_csv(f"{d}/nodes.csv", index=False)

pd.DataFrame([
    ["has_subsidiary", "{}"],
    ["incorporated_in", "{}"],
    ["mentioned_in", "{}"],
], columns=["name", "attributes"]).to_csv(f"{d}/relations.csv", index=False)

pd.DataFrame([
    ["Apple Inc.", "has_subsidiary", "Apple Operations International", "{}"],
    ["Apple Operations International", "incorporated_in", "Ireland", "{}"],
    ["Apple Operations International", "mentioned_in", "AAPL_2025_ex21_c1", "{}"],
    ["Apple Inc.", "mentioned_in", "AAPL_2025_item1_c3", "{}"],
], columns=["source", "relation", "target", "attributes"]).to_csv(f"{d}/edges.csv", index=False)

print("Toy graph created:")
for f in ["nodes.csv", "relations.csv", "edges.csv"]:
    print(" ", f"{d}/{f}")

## 6. Load G-reasoner with local NER + EL and retrieve
`ToyNER` and `ToyEL` do simple string matching, so no OpenAI key is needed.

The `Engine core died` error at the very end is the embedding server shutting down when the script exits. Harmless.

In [ ]:
demo = r"""
from gfmrag import GFMRetriever
from gfmrag.graph_index_construction.ner_model.base_model import BaseNERModel
from gfmrag.graph_index_construction.entity_linking_model.base_model import BaseELModel

class ToyNER(BaseNERModel):
    def __init__(self, **kwargs):
        self.entities = [
            "Apple Inc.",
            "Apple Operations International",
            "Ireland",
        ]
    def __call__(self, query):
        found = []
        for entity in self.entities:
            if entity.lower() in query.lower():
                found.append(entity)
        if "apple" in query.lower() and "Apple Inc." not in found:
            found.append("Apple Inc.")
        return found

class ToyEL(BaseELModel):
    def __init__(self, **kwargs):
        self.entities = []
    def index(self, entity_list):
        self.entities = entity_list
        print("EL indexed entities:")
        for entity in entity_list:
            print("  ", entity)
    def __call__(self, ner_entity_list, topk=1):
        results = {}
        for mention in ner_entity_list:
            matches = []
            for entity in self.entities:
                if mention.lower() == entity.lower():
                    matches.append({"entity": entity, "score": 1.0})
            if not matches:
                for entity in self.entities:
                    if mention.lower() in entity.lower():
                        matches.append({"entity": entity, "score": 0.5})
            results[mention] = matches[:topk]
        return results

ner_model = ToyNER()
el_model = ToyEL()

print("=== Loading G-reasoner ===")
retriever = GFMRetriever.from_index(
    data_dir="./data",
    data_name="toy_fin",
    model_path="rmanluo/G-reasoner-34M",
    ner_model=ner_model,
    el_model=el_model,
)
print("=== G-reasoner loaded successfully ===")

query = "Where is Apple's subsidiary incorporated?"
print("Query:", query)
print("Detected entities:", ner_model(query))

results = retriever.retrieve(query, top_k=5)
print("Raw result:", results)
print("=== Documents ===")
for item in results.get("document", []):
    print(item)
"""
run_py(demo);

---
# Part 2: J&J test for Mohini

Answers her 3 questions:
1. **Can G-reasoner load our own graph?** Cells 8 and 9 load hand-built CSVs, no graph building.
2. **What do the files look like?** Cell 7 prints all three CSVs. Copy those to Mohini.
3. **Numbers as attribute nodes or regular nodes?** Cell 8 tries `type=attribute` (QuadGraph style). Cell 9 is the same graph with the number as a regular `entity`. Compare.

Uses **real J&J chunks** from our `chunks.jsonl`. Needs that file in `/content`.

## 7a. Get chunks.jsonl
If it's not already in `/content`, this opens an upload box. Pick `data/chunks/chunks.jsonl` from the repo. Run all pauses here until you upload.

In [ ]:
import os
CHUNKS = "/content/chunks.jsonl"
if not os.path.exists(CHUNKS):
    from google.colab import files
    print("Upload data/chunks/chunks.jsonl")
    up = files.upload()
    name = list(up.keys())[0]
    if name != "chunks.jsonl":
        os.rename(name, CHUNKS)
print("chunks.jsonl:", os.path.getsize(CHUNKS) // 1024, "KB")

## 7b. Build the J&J graph from real chunks
5 fact edges between entities, one number node, and `mentioned_in` edges to every J&J chunk whose text names that entity. Chunk text goes in the document node's `attributes`.

The revenue value is a **placeholder**. This is a loader test, not a fact.

In [ ]:
import json, os
import pandas as pd

# --- load J&J chunks (field names auto-detected) ---
rows = [json.loads(l) for l in open(CHUNKS)]
keys = rows[0].keys()
ID = next(k for k in ["chunk_id", "id", "uid"] if k in keys)
TEXT = next(k for k in ["text", "chunk_text", "content"] if k in keys)
TICK = next((k for k in ["ticker", "company_ticker"] if k in keys), None)
print("fields:", list(keys))
print(f"using id={ID!r} text={TEXT!r} ticker={TICK!r}")

jnj = [r for r in rows if (r.get(TICK) == "JNJ" if TICK else "JNJ" in str(r[ID]))]
print("J&J chunks:", len(jnj))

# --- entities and fact edges ---
JNJ, IM, MT = "Johnson & Johnson", "Johnson & Johnson Innovative Medicine", "Johnson & Johnson MedTech"
KENVUE, ABIO, SHOCK = "Kenvue", "Abiomed", "Shockwave Medical"
REV = "Johnson & Johnson total revenue FY2025: PLACEHOLDER"   # number node

entities = [JNJ, IM, MT, KENVUE, ABIO, SHOCK]
fact_edges = [
    (JNJ, "has_segment", IM),
    (JNJ, "has_segment", MT),
    (JNJ, "spun_off", KENVUE),
    (MT, "acquired", ABIO),
    (MT, "acquired", SHOCK),
]
num_edge = (JNJ, "has_attribute", REV)

# keyword used to find each entity in chunk text
kw = {JNJ: "Johnson & Johnson", IM: "Innovative Medicine", MT: "MedTech",
      KENVUE: "Kenvue", ABIO: "Abiomed", SHOCK: "Shockwave"}

# link each non-J&J entity to up to 2 chunks that mention it (J&J is in every chunk, so skip it)
MAX_PER = 2
mention_edges, doc_ids = [], []
for e in entities[1:]:
    hits = [r for r in jnj if kw[e].lower() in r[TEXT].lower()][:MAX_PER]
    print(f"{e:40s} found in {len(hits)} chunk(s)")
    for r in hits:
        mention_edges.append((e, "mentioned_in", r[ID]))
        if r[ID] not in doc_ids:
            doc_ids.append(r[ID])
# give J&J itself a link to the docs we kept
for d in doc_ids:
    mention_edges.append((JNJ, "mentioned_in", d))
text_by_id = {r[ID]: r[TEXT] for r in jnj}

relations = {
    "has_segment": "a company has a business segment",
    "spun_off": "a company separated a business into a new independent company",
    "acquired": "a company or segment acquired another company",
    "has_attribute": "an entity has a reported value",
    "mentioned_in": "an entity is mentioned in a text chunk",
}

def write_graph(data_name, number_type):
    d = f"data/{data_name}/processed/stage1"
    os.makedirs(d, exist_ok=True)
    nodes = [[e, "entity", "{}"] for e in entities]
    nodes.append([REV, number_type, "{}"])
    nodes += [[i, "document", repr({"text": text_by_id[i]})] for i in doc_ids]
    pd.DataFrame(nodes, columns=["name", "type", "attributes"]).to_csv(f"{d}/nodes.csv", index=False)
    pd.DataFrame([[k, repr({"description": v})] for k, v in relations.items()],
                 columns=["name", "attributes"]).to_csv(f"{d}/relations.csv", index=False)
    edges = fact_edges + [num_edge] + mention_edges
    pd.DataFrame([[s, r, t, "{}"] for s, r, t in edges],
                 columns=["source", "relation", "target", "attributes"]).to_csv(f"{d}/edges.csv", index=False)
    return d

d_attr = write_graph("jnj_attr", "attribute")   # QuadGraph style
d_ent  = write_graph("jnj_entity", "entity")    # fallback

print(f"\n{len(entities)+1} entity/number nodes, {len(doc_ids)} chunk nodes, "
      f"{len(fact_edges)} fact edges + 1 number edge + {len(mention_edges)} mentioned_in edges")

# --- print example files for Mohini (Q2) ---
pd.set_option("display.max_colwidth", 80)
for f in ["nodes.csv", "relations.csv", "edges.csv"]:
    print(f"\n===== {f} (jnj_attr) =====")
    print(pd.read_csv(f"{d_attr}/{f}").to_string(index=False))

## 7c. Retrieval script used by cells 8 and 9
Same `ToyNER` / `ToyEL` as part 1, with the J&J names. `J&J` in a query is expanded to `Johnson & Johnson`.

In [ ]:
NAMES = entities  # from 7b

JNJ_SCRIPT = r"""
import traceback
from gfmrag import GFMRetriever
from gfmrag.graph_index_construction.ner_model.base_model import BaseNERModel
from gfmrag.graph_index_construction.entity_linking_model.base_model import BaseELModel

NAMES = __NAMES__
DATA = "__DATA__"
TYPES = __TYPES__

class ToyNER(BaseNERModel):
    def __init__(self, **kwargs):
        self.entities = NAMES
    def __call__(self, query):
        q = query.replace("J&J", "Johnson & Johnson").lower()
        return [e for e in self.entities if e.lower() in q]

class ToyEL(BaseELModel):
    def __init__(self, **kwargs):
        self.entities = []
    def index(self, entity_list):
        self.entities = entity_list
    def __call__(self, ner_entity_list, topk=1):
        out = {}
        for m in ner_entity_list:
            hits = [{"entity": e, "score": 1.0} for e in self.entities if m.lower() == e.lower()]
            if not hits:
                hits = [{"entity": e, "score": 0.5} for e in self.entities if m.lower() in e.lower()]
            out[m] = hits[:topk]
        return out

ner = ToyNER()
print(f"=== Loading G-reasoner on {DATA} ===")
r = GFMRetriever.from_index(
    data_dir="./data", data_name=DATA,
    model_path="rmanluo/G-reasoner-34M",
    ner_model=ner, el_model=ToyEL(),
)
print("=== G-reasoner loaded successfully ===")
print("node types in graph:", list(r.graph.nodes_by_type.keys()))

for q in ["What did J&J spin off?",
          "Which companies did Johnson & Johnson MedTech acquire?",
          "What was J&J total revenue?"]:
    print("\n### Query:", q)
    print("NER:", ner(q))
    for t in TYPES:
        try:
            res = r.retrieve(q, top_k=3, target_types=[t])
            print(f"  top {t}:")
            for it in res[t]:
                print(f"    {it['score']:8.3f}  {it['id'][:70]}")
        except Exception as e:
            print(f"  {t}: FAILED -> {type(e).__name__}: {e}")
"""

def jnj_script(data, types):
    return (JNJ_SCRIPT.replace("__NAMES__", repr(NAMES))
                      .replace("__DATA__", data)
                      .replace("__TYPES__", repr(types)))
print("script ready")

## 8. Test A: number as `attribute` node (QuadGraph style)
- Loads and `attribute` shows up in node types → attribute nodes work, go QuadGraph style.
- Fails on load → the released loader doesn't support `attribute`. Cell 9 is the fallback.

In [ ]:
resA = run_py(jnj_script("jnj_attr", ["document", "entity", "attribute"]))

## 9. Test B: number as regular `entity` node (fallback)
Same graph, only the number node's type changes.

In [ ]:
resB = run_py(jnj_script("jnj_entity", ["document", "entity"]))

## 10. Summary to send Mohini

In [ ]:
ok = lambda r: r.returncode == 0 and "loaded successfully" in r.stdout
a_ok = ok(resA) and "'attribute'" in resA.stdout.split("node types in graph:")[-1].split("\n")[0]
print("Q1  Loads our own CSVs (no graph building):", "YES" if ok(resA) or ok(resB) else "NO, see errors")
print("Q2  Example files: printed in cell 7b")
print("Q3  attribute-type nodes supported:", "YES, go QuadGraph style" if a_ok else
      ("NO, use regular entity nodes" if ok(resB) else "unclear, see errors"))